# Decision-robust allocation deployment

Unexecuted usage examples for the existing framework and new patient_assignment commands. Run on the intended compute node/kernel from the repository root. No clinical records or model outputs are embedded. The original Deployment_NEW.ipynb was absent from this checkout, so this notebook provides the allocation setup/workflow only.

For an existing environment: `python -m pip install gurobipy`; or install the package with `python -m pip install -e '.[optimization]'`. Existing inference requires its existing GPU/model dependencies. All output directories should be fresh.

## Existing file-license setup

This cell is specific to your reported `/home/pathilda/gurobi.lic` deployment example. It respects an existing GRB_LICENSE_FILE and checks existence/readability without showing contents. Run it before any Gurobi environment/model. The reusable package uses standard discovery and does not require this path. Validity and compute-node compatibility still need verification. No activation or new license retrieval is performed.

In [ ]:
import os
from pathlib import Path

license_path = Path(os.environ.get("GRB_LICENSE_FILE", "/home/pathilda/gurobi.lic"))
if not license_path.is_file():
    raise FileNotFoundError(
        f"Gurobi license file not found: {license_path}. "
        "Set GRB_LICENSE_FILE to your actual gurobi.lic path."
    )
if not os.access(license_path, os.R_OK):
    raise PermissionError(f"Gurobi license file is not readable: {license_path}")
os.environ["GRB_LICENSE_FILE"] = str(license_path)

## Optional synthetic license smoke test

Run explicitly on the actual compute node. Expected objective: 1. Inspect the startup license message locally; a tiny solve alone does not prove a full license (gurobipy can have a size-limited license). Do not copy license contents or secrets into reports.

In [ ]:
import gurobipy as gp
from gurobipy import GRB

with gp.Env() as env, gp.Model(env=env) as model:
    x = model.addVar(vtype=GRB.BINARY)
    model.setObjective(x, GRB.MAXIMIZE)
    model.optimize()
    if model.Status != GRB.OPTIMAL:
        raise RuntimeError(f"Gurobi smoke test failed, status={model.Status}")
    print("Gurobi smoke test passed; objective:", model.ObjVal)

## Create reusable discipline membership

Chunking reads cohort metadata only, with no note/de-identification requirement. Literal case-sensitive space-delimited tags; repeated disciplines select a union. First source row determines both notes and tags. Chunk seed 42 is independent of solver and selection seeds. Smaller remainder chunks are retained.

In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable, "run.py", "chunk", "--input", "patients.xlsx",
    "--discipline", "SLP", "--chunk-size", "50", "--seed", "42",
    "--output", "scenarios/slp_50_seed42",
], check=True)

## Explicit inference or reuse existing predictions

Skip this cell if suitable fully covered predictions already exist. This cell launches real inference only when you explicitly run it. All patients in membership need valid predictions. Fresh output directories avoid the historical stale selected-record export risk; checkpoint migration is outside this enhancement.

In [ ]:
RUN_INFERENCE = False  # Change only when ready to run on approved de-identified inputs.
if RUN_INFERENCE:
    for model in ("qwen", "medgemma"):
        subprocess.run([
            sys.executable, "run.py", "--pipeline", "classify",
            "--input", "patients.xlsx", "--discipline", "SLP", "--model", model,
            "--deidentified-confirmed", "--output", f"results/{model}_run1",
        ], check=True)

## Solve independent chunks

Each chunk restarts at the same initial state. 35 capacity and initial counts 12,10,14 leave 69 slots, enough for 50 new patients. Final count balance is optimized first, then final severity balance. Initial loads are in the same experimental difficulty-weight units as patient weights 1/2/3.

In [ ]:
operational = [
    "--chunks", "scenarios/slp_50_seed42/chunks.csv",
    "--clinician", "A", "12", "24", "--clinician", "B", "10", "27",
    "--clinician", "C", "14", "22", "--clinician-capacity", "35",
    "--threads", "1", "--solver-seed", "0",
]
subprocess.run([
    sys.executable, "run.py", "solve", *operational,
    "--predictions", "results/qwen_run1/qwen/client_classifications.xlsx",
    "--label", "QS", "--output", "assignments/QS_run1",
], check=True)

## Two-model matched comparison

Both sides can move inside their certified lexicographic optimal sets. Clinician identities stay fixed. The closest pair is solver-selected without a regret or ID tie-break. The same model may receive a different pair-specific allocation against another partner.

In [ ]:
ai_input = [
    "--predictions-a", "results/qwen_run1/qwen/client_classifications.xlsx", "--label-a", "QS",
]
subprocess.run([
    sys.executable, "run.py", "compare-assignments", *operational, *ai_input,
    "--predictions-b", "results/medgemma_run1/medgemma/client_classifications.xlsx", "--label-b", "MS",
    "--selection-mode", "matched", "--output", "comparisons/QS_vs_MS_run1",
], check=True)

## AI versus explicit reference, both selection modes

Reference files use the exact existing category labels. Every member must be labeled; explicitly prepare a restricted manifest beforehand for partially labeled reference data. Saved AI assignments are evaluated with reference weights and initial severity; the denominator is reference severity optimum subject to count optimum.

Matched regret is retrospective and matching-conditioned, using reference information; it is not deployment-policy performance without reference labels. Minimum disagreement does not imply minimum regret. Random selection is independent per model and uses continuous random objectives within optimal sets; it is not uniform sampling of allocations. All repeats are reported, not treated as independent clinical samples.

In [ ]:
reference_input = ["--predictions-b", "reference.xlsx", "--label-b", "Reference", "--reference-side", "B"]
subprocess.run([
    sys.executable, "run.py", "compare-assignments", *operational, *ai_input, *reference_input,
    "--selection-mode", "matched", "--output", "comparisons/QS_vs_reference_matched_run1",
], check=True)

In [ ]:
subprocess.run([
    sys.executable, "run.py", "compare-assignments", *operational, *ai_input, *reference_input,
    "--selection-mode", "random", "--selection-seed", "42", "--selection-repeats", "20",
    "--output", "comparisons/QS_vs_reference_random_run1",
], check=True)

## Licensing and verification notes

The gurobipy Python installation does not include grbgetkey. Named-User activation uses official license tools and the exact portal command on the intended machine; academic retrieval requires the academic network. WLS uses its downloaded/generated file, not grbgetkey, and requires connectivity to obtain tokens. Confirm institutional license type when HPC nodes change. Never share credentials in run reports.

Run `python -m pytest patient_assignment/tests -q -rs` on this node. License-unavailable optimization tests skip explicitly, and skipped tests are unverified. Exit codes are 0 for certified completion, 1 for incomplete optimization, and 2 for input/config/dependency errors. Provisional candidates do not enter certified summary aggregates.

See [package guide](patient_assignment/README.md), [Python setup](https://support.gurobi.com/hc/en-us/articles/360044290292-How-do-I-install-Gurobi-for-Python), and [WLS setup](https://support.gurobi.com/hc/en-us/articles/13232844297489-How-do-I-set-up-a-Web-License-Service-WLS-license).